# 🐝 Swarm Tuning — Kaggle Worker

This notebook is one **worker** in a federated swarm. It repeatedly:

1. pulls the global model weights from the Hugging Face Space (parameter server),
2. trains on one micro-batch (computing gradients),
3. pushes **only the gradients** back.

Run several copies (different `WORKER_ID` / `SHARD`, your friends' notebooks) to
form the swarm. Set `SERVER_URL` to your deployed Space and, if your Space uses a
token, store it as a Kaggle **Secret** named `SWARM_TOKEN`.

In [ ]:
# Install the swarm package (torch is already present on Kaggle images).
!pip install -q "git+https://github.com/aabhimittal/swarm-tuning-kernel-federated.git@claude/federated-swarm-training-bw547m"

In [ ]:
# ---- Configuration -------------------------------------------------------
SERVER_URL = "https://abhimittal-swarm-server.hf.space"  # the deployed parameter-server Space
WORKER_ID  = "kaggle-1"
SHARD       = 0      # this worker's data shard index
NUM_SHARDS  = 4      # total workers in the swarm
STEPS       = 200    # pull/train/push iterations
BATCH_SIZE  = 16

# Bearer token (the Space requires SWARM_TOKEN). Store it as a Kaggle Secret
# named "SWARM_TOKEN", or paste it directly into TOKEN below.
TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("SWARM_TOKEN")
except Exception:
    pass

In [ ]:
from worker.client import SwarmClient

client = SwarmClient(
    server_url=SERVER_URL,
    worker_id=WORKER_ID,
    batch_size=BATCH_SIZE,
    shard=SHARD,
    num_shards=NUM_SHARDS,
    token=TOKEN,
)
print(f"Connected. model params={client.model.num_params():,} vocab={client.model_cfg.vocab_size}")

client.run(steps=STEPS)
client.close()

Watch the global model advance at `SERVER_URL/status` while this runs. In `sync`
mode the server takes one optimizer step every `SWARM_WORLD_SIZE` gradients — so
the global batch is the *sum* of every worker's micro-batch. That's the whole
trick: a model trained by many small instances cooperating over HTTP.